## データベースオブジェクトの準備

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ontobricks;
CREATE SCHEMA IF NOT EXISTS ontobricks.action_demo;

In [0]:
%sql
CREATE OR REPLACE TABLE ontobricks.action_demo.customer (
    customer_id STRING,
    customer_name STRING
);

INSERT INTO ontobricks.action_demo.customer
VALUES
    ('C001', 'Alice');

In [0]:
%sql
SELECT *
FROM ontobricks.action_demo.customer;

In [0]:
%sql
CREATE OR REPLACE FUNCTION
    ontobricks.action_demo.confirm_customer_action(
        p_customer_id STRING
    )
RETURNS STRING
COMMENT 'Confirm that the action was invoked for this customer'
RETURN concat(
    'ACTION_OK: ',
    p_customer_id
);

In [0]:
%sql
SELECT
    ontobricks.action_demo.confirm_customer_action('C001');

In [0]:
%sql
CREATE OR REPLACE FUNCTION
    ontobricks.action_demo.get_customer_name_action(
        p_customer_id STRING
    )
RETURNS STRING
LANGUAGE SQL
COMMENT 'Get the customer name'
READS SQL DATA
RETURN
    SELECT MAX(c.customer_name)
    FROM ontobricks.action_demo.customer AS c
    WHERE c.customer_id = p_customer_id;

In [0]:
%sql
SELECT
    ontobricks.action_demo.get_customer_name_action('C001');

In [0]:
%sql
CREATE OR REPLACE FUNCTION
    ontobricks.action_demo.get_customer_detail_action(
        p_customer_id STRING
    )
RETURNS TABLE (
    customer_id STRING,
    customer_name STRING
)
COMMENT 'Get customer details'
READS SQL DATA
RETURN
    SELECT
        c.customer_id,
        c.customer_name
    FROM ontobricks.action_demo.customer AS c
    WHERE c.customer_id = p_customer_id;

In [0]:
%sql
SELECT *
FROM ontobricks.action_demo.get_customer_detail_action('C001');

### Databricks Apps に対する権限付与

In [0]:
%sql
-- OntoBricks App の Service Principal Application ID
DECLARE OR REPLACE VARIABLE ontobricks_app_principal STRING
DEFAULT 'cb683fcf-1151-4d99-92f8-5f6982d6093c';

EXECUTE IMMEDIATE
  'GRANT USE CATALOG ON CATALOG ontobricks TO `' 
  || ontobricks_app_principal 
  || '`';

EXECUTE IMMEDIATE
  'GRANT USE SCHEMA ON SCHEMA ontobricks.action_demo TO `' 
  || ontobricks_app_principal 
  || '`';

EXECUTE IMMEDIATE
  'GRANT SELECT ON SCHEMA ontobricks.action_demo TO `' 
  || ontobricks_app_principal 
  || '`';

EXECUTE IMMEDIATE
  'GRANT EXECUTE ON SCHEMA ontobricks.action_demo TO `'
  || ontobricks_app_principal
  || '`';

EOF